# 05 — Estrategia de validación (§9 del README)

| | |
|---|---|
| **Entrada** | Candidatos de §8 (`notebooks/04_models.ipynb`) |
| **Objetivo** | Validación cruzada reproducible, **métrica principal** y **umbral** de decisión |
| **Salida** | Resultados de CV (media/desvío) y umbral elegido para README §9 |

**Reglas:** sin resultados inventados; el preprocesador se ajusta **dentro de cada fold** (viaja en el `Pipeline`).

## 1. Diseño de la validación

- **Partición train/test** (de §6): 80/20 estratificada, `random_state=42`. El **test queda reservado** para la evaluación final en §10 y no se usa aquí.
- **Validación cruzada**: `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` sobre el **train**. Cada fold conserva ~11,3% de `yes`.
- **Prevención de fuga**: el preprocesador + modelo son un único `Pipeline`; `cross_validate` re-ajusta todo en cada fold. `duplicados` ya fueron eliminados antes del split (§6).
- **Métricas**: ROC-AUC, PR-AUC, precision, recall, F1 (clase `yes`).
- **Métrica principal (decisión justificada)**: **PR-AUC**. Razones: (1) con desbalance 7,9:1 el accuracy engaña; (2) PR-AUC resume el desempeño sobre la clase `yes` a lo largo de todos los umbrales; (3) es insensible al gran número de verdaderos negativos. Se reporta ROC-AUC como complemento y F1/recall en el umbral elegido.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate

from src.models import build_candidate_models
from src.preprocessing import (
    clean_data,
    load_raw_data,
    split_features_target,
    train_test_split_data,
)

df = clean_data(load_raw_data())
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = train_test_split_data(X, y)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"CV: {cv.get_n_splits()} folds sobre {X_train.shape[0]:,} filas de train")
print(f"Test reservado: {X_test.shape[0]:,} filas ({y_test.mean() * 100:.2f}% yes)")

CV: 5 folds sobre 32,940 filas de train
Test reservado: 8,236 filas (11.27% yes)


## 2. Validación cruzada de los candidatos

Se usa `cross_validate` con el `Pipeline` completo como estimador (evita fuga) y varias métricas a la vez.

In [2]:
scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "accuracy": "accuracy",
}

modelos = build_candidate_models()

filas = []
for nombre, modelo in modelos.items():
    res = cross_validate(
        modelo, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1
    )
    fila = {"Modelo": nombre}
    for metrica in scoring:
        clave = f"test_{metrica}"
        fila[f"{metrica}_mean"] = res[clave].mean()
        fila[f"{metrica}_std"] = res[clave].std()
    fila["t_fit_s"] = res["fit_time"].mean()
    filas.append(fila)

cv_tabla = pd.DataFrame(filas).set_index("Modelo").sort_values("pr_auc_mean", ascending=False)
cols = ["pr_auc_mean", "pr_auc_std", "roc_auc_mean", "roc_auc_std",
        "precision_mean", "recall_mean", "f1_mean", "accuracy_mean", "t_fit_s"]
cv_tabla[cols].round(3)

,pr_auc_mean,pr_auc_std,roc_auc_mean,roc_auc_std,precision_mean,recall_mean,f1_mean,accuracy_mean,t_fit_s
Modelo,,,,,,,,,
Gradient Boosting,0.447,0.014,0.793,0.011,0.662,0.215,0.324,0.899,15.854
Random Forest,0.441,0.016,0.786,0.009,0.410,0.589,0.484,0.858,20.185
Regresion logistica,0.421,0.013,0.760,0.011,0.699,0.182,0.289,0.899,0.825
Regresion logistica balanceada,0.420,0.013,0.761,0.011,0.277,0.672,0.393,0.766,0.697
Arbol de decision,0.397,0.017,0.775,0.012,0.644,0.214,0.321,0.898,0.434
KNN,0.385,0.014,0.753,0.012,0.623,0.208,0.312,0.897,0.154


In [3]:
print("Resultados de validacion cruzada (5 folds), ordenados por PR-AUC:")
print(cv_tabla[cols].round(3).to_string())
print("\nMetrica principal = PR-AUC (desbalance 7,9:1).")
print("El desvio (std) indica estabilidad entre folds.")

Resultados de validacion cruzada (5 folds), ordenados por PR-AUC:
                                pr_auc_mean  pr_auc_std  roc_auc_mean  roc_auc_std  precision_mean  recall_mean  f1_mean  accuracy_mean  t_fit_s
Modelo                                                                                                                                          
Gradient Boosting                     0.447       0.014         0.793        0.011           0.662        0.215    0.324          0.899   15.854
Random Forest                         0.441       0.016         0.786        0.009           0.410        0.589    0.484          0.858   20.185
Regresion logistica                   0.421       0.013         0.760        0.011           0.699        0.182    0.289          0.899    0.825
Regresion logistica balanceada        0.420       0.013         0.761        0.011           0.277        0.672    0.393          0.766    0.697
Arbol de decision                     0.397       0.017         

## 3. Elección de la métrica principal y del umbral

- **Métrica principal**: **PR-AUC** (§9, justificada arriba). El mejor modelo por PR-AUC medio será el modelo seleccionado, salvo que otro sea casi igual y mucho mejor en interpretabilidad/estabilidad.
- **Umbral**: el umbral 0,5 no es óptimo con clases desbalanceadas. Se elige el umbral que **maximiza F1** en la clase `yes`, calculado con validación cruzada (out-of-fold) para no usar el test.
- Se reporta también el umbral que prioriza **recall** (contexto: falso negativo más costoso, README §7).

In [4]:
from sklearn.model_selection import cross_val_predict

mejor_nombre = cv_tabla.index[0]
print("Mejor modelo por PR-AUC en CV:", mejor_nombre)

modelo_final = build_candidate_models()[mejor_nombre]

# Probabilidades out-of-fold (cada muestra predicha por un modelo que no la vio)
y_oof = cross_val_predict(
    modelo_final, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]

prec, rec, umbrales = precision_recall_curve(y_train, y_oof)
f1s = np.divide(2 * prec * rec, (prec + rec), out=np.zeros_like(prec), where=(prec + rec) > 0)
idx_f1 = int(np.argmax(f1s[:-1]))
umbral_f1 = float(umbrales[idx_f1])

print(f"PR-AUC out-of-fold: {average_precision_score(y_train, y_oof):.3f}")
print(f"Umbral que maximiza F1: {umbral_f1:.3f} -> F1={f1s[idx_f1]:.3f}, "
      f"precision={prec[idx_f1]:.3f}, recall={rec[idx_f1]:.3f}")

mask_at_05 = np.argmin(np.abs(umbrales - 0.5))
print(f"Umbral 0.5 (referencia)   -> F1={f1s[mask_at_05]:.3f}, "
      f"precision={prec[mask_at_05]:.3f}, recall={rec[mask_at_05]:.3f}")

Mejor modelo por PR-AUC en CV: Gradient Boosting


PR-AUC out-of-fold: 0.444
Umbral que maximiza F1: 0.177 -> F1=0.493, precision=0.431, recall=0.576
Umbral 0.5 (referencia)   -> F1=0.324, precision=0.660, recall=0.215


## 4. Umbral orientado a recall

El proyecto prioriza no perder clientes que contratarían (falso negativo costoso). Se busca el umbral con el mayor recall cuya precision no caiga por debajo de un piso razonable (p. ej. 0,30), documentando el trade-off.

In [5]:
piso_precision = 0.30
validos = prec[:-1] >= piso_precision
if validos.any():
    idx_rec = int(np.argmax(np.where(validos, rec[:-1], -1)))
    umbral_rec = float(umbrales[idx_rec])
    print(f"Umbral orientado a recall (precision >= {piso_precision}):")
    print(f"  umbral={umbral_rec:.3f}, recall={rec[idx_rec]:.3f}, precision={prec[idx_rec]:.3f}, f1={f1s[idx_rec]:.3f}")
else:
    print("Ningun umbral alcanza el piso de precision indicado.")

Umbral orientado a recall (precision >= 0.3):
  umbral=0.085, recall=0.673, precision=0.300, f1=0.415


## 5. Resumen de §9

| Punto del README §9 | Definición |
|---|---|
| División train/test | 80/20 estratificada, `random_state=42` |
| Estratificación | Sí (11,27% yes en cada parte) |
| Validación cruzada | `StratifiedKFold(5, shuffle=True, random_state=42)` |
| Prevención de leakage | Pipeline completo re-ajustado por fold; duplicados ya eliminados |
| Métricas | PR-AUC (principal), ROC-AUC, precision, recall, F1 |
| Métrica principal | **PR-AUC** (desbalance; desempeño sobre la clase `yes`) |

El umbral elegido y el modelo seleccionado se aplican al **test reservado** en §10.

In [6]:
resultado_umbral = {
    "modelo_seleccionado": mejor_nombre,
    "umbral_f1": umbral_f1,
    "pr_auc_oof": float(average_precision_score(y_train, y_oof)),
}
print(resultado_umbral)

{'modelo_seleccionado': 'Gradient Boosting', 'umbral_f1': 0.1767441125685916, 'pr_auc_oof': 0.44392905841942937}
